# Multi-Faceted Filtering — Metadata-Driven Retrieval Refinement

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/multi_faceted_filtering.ipynb)

Real-world RAG systems don't retrieve by semantic similarity alone. Documents carry **structured metadata** — dates, categories, access levels, source types — that must be filtered before or after semantic search. This notebook builds a retrieval system that combines metadata filtering with vector similarity.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **why pure semantic search fails** in production (stale docs, wrong department, security)
- Implement **pre-filtering** (filter then retrieve) and **post-filtering** (retrieve then filter) strategies
- Build a **metadata-enriched FAISS index** with structured filter support
- Use an **LLM to extract filter criteria** from natural language queries
- Evaluate **filtering impact** on precision and recall

## Part 1 — Why Pure Semantic Search Isn't Enough

Consider a legal firm's RAG system. A lawyer asks: "What are our data retention policies updated after 2023?"

Pure semantic search returns the 10 most semantically similar documents. But some might be:
- **2019 policies** (outdated, superseded by newer versions)
- **Client-specific policies** the lawyer doesn't have access to
- **Draft documents** not yet approved
- **Blog posts about retention** (wrong document type)

All of these are *semantically relevant* but *practically wrong*. The system needs to filter on:

| Metadata Field | Type | Example Filter |
|---|---|---|
| date | Temporal | After 2023-01-01 |
| department | Categorical | Legal, Compliance |
| status | Categorical | Published (not Draft) |
| ccess_level | Hierarchical | user's clearance >= doc's level |
| doc_type | Categorical | Policy (not Blog) |

### Pre-Filtering vs Post-Filtering

**Pre-filtering**: Apply metadata filters first, then run semantic search on the filtered subset.
- Pro: Guaranteed compliance (no unauthorized docs ever retrieved)
- Con: Might filter too aggressively, leaving too few docs for good semantic search

**Post-filtering**: Run semantic search first (retrieve top-K), then filter results.
- Pro: Better semantic diversity (larger search pool)
- Con: Might return fewer than K results after filtering; unauthorized docs briefly in memory

## Part 2 — Environment Setup

In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
!pip install -q sentence-transformers faiss-cpu datasets transformers torch bitsandbytes accelerate
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# Cell 2: Create a synthetic document collection with metadata
# ============================================================
import numpy as np
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

# Synthetic documents with rich metadata
DEPARTMENTS = ['Engineering', 'Legal', 'Marketing', 'Finance', 'HR']
DOC_TYPES = ['Policy', 'Report', 'Memo', 'Guide', 'FAQ']
STATUS = ['Published', 'Draft', 'Archived']
ACCESS_LEVELS = [1, 2, 3]  # 1=public, 2=internal, 3=confidential

documents = []
for i in range(200):
    dept = random.choice(DEPARTMENTS)
    days_ago = random.randint(0, 1500)
    doc = {
        'id': f'doc_{i:03d}',
        'title': f'{dept} {random.choice(DOC_TYPES)} #{i}',
        'content': f'This document from {dept} covers topic area {i % 20}. ' * random.randint(3, 8),
        'department': dept,
        'doc_type': random.choice(DOC_TYPES),
        'status': random.choices(STATUS, weights=[0.6, 0.2, 0.2])[0],
        'access_level': random.choice(ACCESS_LEVELS),
        'date': (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d'),
    }
    documents.append(doc)

print(f'Created {len(documents)} documents')
print(f'Departments: {set(d["department"] for d in documents)}')
print(f'Date range: {min(d["date"] for d in documents)} to {max(d["date"] for d in documents)}')

## Part 3 — Building the Metadata-Enriched Index

In [ ]:
# ============================================================
# Cell 3: Embed documents and build FAISS index
# ============================================================
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all documents
texts = [d['content'] for d in documents]
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

# Build FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product (cosine after normalization)
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f'Index built: {index.ntotal} vectors, dim={dim}')

In [ ]:
# ============================================================
# Cell 4: Implement pre-filter and post-filter strategies
# ============================================================

def apply_metadata_filters(docs, filters):
    """Filter documents based on metadata criteria."""
    filtered = docs
    for field, condition in filters.items():
        if field == 'date_after':
            filtered = [d for d in filtered if d['date'] >= condition]
        elif field == 'date_before':
            filtered = [d for d in filtered if d['date'] <= condition]
        elif field == 'max_access_level':
            filtered = [d for d in filtered if d['access_level'] <= condition]
        elif isinstance(condition, list):
            filtered = [d for d in filtered if d.get(field) in condition]
        else:
            filtered = [d for d in filtered if d.get(field) == condition]
    return filtered

def pre_filter_retrieve(query, filters, k=5):
    """Filter first, then semantic search on subset."""
    # Step 1: metadata filter
    filtered_docs = apply_metadata_filters(documents, filters)
    if not filtered_docs:
        return [], 'pre-filter'
    
    # Step 2: embed only filtered docs and search
    filtered_texts = [d['content'] for d in filtered_docs]
    filtered_embeddings = embed_model.encode(filtered_texts, convert_to_numpy=True)
    faiss.normalize_L2(filtered_embeddings)
    
    temp_index = faiss.IndexFlatIP(filtered_embeddings.shape[1])
    temp_index.add(filtered_embeddings)
    
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = temp_index.search(q_emb, min(k, len(filtered_docs)))
    
    results = [(filtered_docs[idx], float(scores[0][j])) for j, idx in enumerate(indices[0])]
    return results, 'pre-filter'

def post_filter_retrieve(query, filters, k=5, oversample=3):
    """Semantic search first (oversampled), then filter."""
    # Step 1: retrieve top-K*oversample
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, k * oversample)
    
    candidates = [(documents[idx], float(scores[0][j])) for j, idx in enumerate(indices[0])]
    
    # Step 2: apply metadata filters
    filtered = [(doc, score) for doc, score in candidates 
                if doc in apply_metadata_filters([doc], filters)]
    
    return filtered[:k], 'post-filter'

print('Pre-filter and post-filter strategies ready!')

## Part 4 — Comparing Strategies

In [ ]:
# ============================================================
# Cell 5: Compare pre-filter vs post-filter
# ============================================================

query = 'data retention and compliance policies'
filters = {
    'department': ['Legal', 'Finance'],
    'status': 'Published',
    'date_after': '2024-01-01',
    'max_access_level': 2
}

print('Query:', query)
print('Filters:', filters)
print()

for strategy_fn, name in [(pre_filter_retrieve, 'PRE-FILTER'), (post_filter_retrieve, 'POST-FILTER')]:
    results, strategy = strategy_fn(query, filters, k=5)
    print(f'\n--- {name} (returned {len(results)} results) ---')
    for doc, score in results:
        print(f'  [{score:.3f}] {doc["title"]} | {doc["department"]} | {doc["status"]} | {doc["date"]} | L{doc["access_level"]}')


## Part 5 — Dynamic Filter Generation with LLM

Instead of requiring users to manually specify filters, we can use an LLM to **extract filter criteria from natural language queries**. "Show me published engineering guides from this year" should automatically become {department: Engineering, doc_type: Guide, status: Published, date_after: 2025-01-01}.

In [ ]:
# ============================================================
# Cell 6: Load LLM for filter extraction
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json as json_mod

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-8B', quantization_config=bnb_config, device_map='auto')
llm_tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-8B')
print('LLM loaded for filter extraction')

In [ ]:
# ============================================================
# Cell 7: Extract filters from natural language
# ============================================================
import re, json as json_mod

def extract_filters(nl_query):
    prompt = f"""Extract metadata filters from this search query. Available fields:
- department: one of [Engineering, Legal, Marketing, Finance, HR]
- doc_type: one of [Policy, Report, Memo, Guide, FAQ]
- status: one of [Published, Draft, Archived]
- date_after: YYYY-MM-DD format
- date_before: YYYY-MM-DD format
- max_access_level: 1 (public), 2 (internal), 3 (confidential)

Query: \"{nl_query}\"

Return ONLY a JSON object with the applicable filters. Example:
{{"department": ["Legal"], "status": "Published", "date_after": "2024-01-01"}}

JSON:"""
    messages = [{"role": "user", "content": prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok(text, return_tensors='pt').to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
    response = llm_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    # Parse JSON from response
    try:
        match = re.search(r'\{[^}]+\}', response)
        if match:
            return json_mod.loads(match.group())
    except:
        pass
    return {}

# Test natural language filter extraction
test_queries = [
    'Show me published engineering guides from 2024',
    'Find all confidential legal policies',
    'Recent marketing reports'
]

for q in test_queries:
    filters = extract_filters(q)
    print(f'Query: "{q}"')
    print(f'  Extracted filters: {filters}\n')

## Part 6 — Evaluating Filter Impact

Filtering improves **precision** (fewer irrelevant results) but can hurt **recall** (might exclude relevant docs). Let's measure this tradeoff.

In [ ]:
# ============================================================
# Cell 8: Measure filtering impact on retrieval quality
# ============================================================

def precision_at_k(results, relevant_depts, k=5):
    """What fraction of top-k results are from relevant departments?"""
    hits = sum(1 for doc, _ in results[:k] if doc['department'] in relevant_depts)
    return hits / min(k, len(results)) if results else 0.0

query = 'compliance and audit procedures'
relevant_depts = ['Legal', 'Finance']

# Unfiltered retrieval
q_emb = embed_model.encode([query], convert_to_numpy=True)
faiss.normalize_L2(q_emb)
scores, indices = index.search(q_emb, 10)
unfiltered = [(documents[idx], float(scores[0][j])) for j, idx in enumerate(indices[0])]

# Filtered retrieval
filtered_results, _ = pre_filter_retrieve(query, {'department': relevant_depts, 'status': 'Published'}, k=10)

print('Unfiltered retrieval:')
print(f'  Precision@5 (dept match): {precision_at_k(unfiltered, relevant_depts):.2f}')
for doc, score in unfiltered[:5]:
    marker = 'OK' if doc['department'] in relevant_depts else 'XX'
    print(f'  [{marker}] {score:.3f} {doc["title"]} | {doc["department"]}')

print(f'\nFiltered retrieval:')
print(f'  Precision@5 (dept match): {precision_at_k(filtered_results, relevant_depts):.2f}')
for doc, score in filtered_results[:5]:
    marker = 'OK' if doc['department'] in relevant_depts else 'XX'
    print(f'  [{marker}] {score:.3f} {doc["title"]} | {doc["department"]}')

## 🏋️ Exercises

**Exercise 1 — Hierarchical Filters:** Implement a hierarchical department filter where filtering on "Engineering" also includes sub-departments like "DevOps" and "QA". Add sub-department metadata to the documents and test.

**Exercise 2 — Adaptive Oversampling:** The post-filter strategy uses a fixed 3x oversample. Implement an adaptive version that increases oversampling when filters are very restrictive (few docs match). Measure the recall improvement.

**Exercise 3 — Filter Relaxation:** When pre-filtering returns too few results (<3), implement a filter relaxation strategy that progressively removes the least important filters. Define a filter priority ordering and test.

## 📝 Key Takeaways

- **Pure semantic search fails in production** — metadata filtering is essential for access control, freshness, and relevance
- **Pre-filtering guarantees compliance** but may reduce search quality; **post-filtering preserves semantic diversity** but risks unauthorized access
- **Dynamic filter extraction** using an LLM turns natural language into structured queries
- **Oversampling** compensates for post-filter losses — retrieve K×N then filter to K
- **Measure the tradeoff**: filtering improves precision but can hurt recall

## 📚 References & Further Reading

- [Metadata Filtering in Vector Databases](https://www.pinecone.io/learn/metadata-filtering/) — Pinecone's guide
- [Hybrid Search: Combining Filters with Vectors](https://weaviate.io/developers/weaviate/search/filters) — Weaviate docs
- Related notebooks: [fusion_retrieval.ipynb](fusion_retrieval.ipynb), [adaptive_retrieval.ipynb](adaptive_retrieval.ipynb)